# NYISO Analytics Pipeline

This notebook demonstrates the complete PySpark pipeline for NYISO electricity market analysis:

1. **Data Ingestion** - Load price and load data
2. **Data Processing** - Clean, merge, and transform
3. **Feature Engineering** - Create ML features
4. **Model Training** - Train prediction models
5. **Evaluation** - Assess model performance

In [ ]:
# Setup
import os
import sys

# Set JAVA_HOME
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

# Add project root to path
project_root = os.path.dirname(os.getcwd())
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

In [ ]:
# Import project modules
from src.utils import get_spark_session, CONFIG
from src.ingestion import NYISODataLoader
from src.processing import NYISODataProcessor
from src.features import NYISOFeatureEngineer
from src.models import PricePredictor, DemandForecaster, SpikeDetector
from src.models.evaluator import ModelEvaluator

# Create Spark session
spark = get_spark_session()
print(f"Spark version: {spark.version}")

## 1. Data Ingestion

In [ ]:
# Initialize data loader
loader = NYISODataLoader(spark)

# Load price and load data
price_df, load_df = loader.load_all()

print(f"Price records: {price_df.count():,}")
print(f"Load records: {load_df.count():,}")

In [ ]:
# Examine price data
print("Price Data Schema:")
price_df.printSchema()
price_df.show(5)

In [ ]:
# Examine load data
print("Load Data Schema:")
load_df.printSchema()
load_df.show(5)

## 2. Data Processing

In [ ]:
# Initialize processor
processor = NYISODataProcessor(spark)

# Run full processing pipeline
processed_df = processor.process_all(price_df, load_df)

print(f"Processed records: {processed_df.count():,}")
print(f"Columns: {len(processed_df.columns)}")

In [ ]:
# View processed data
processed_df.select(
    "timestamp", "Name", "LBMP_avg", "Load_MW", 
    "hour", "day_of_week", "month", "is_weekend"
).show(10)

## 3. Feature Engineering

In [ ]:
# Initialize feature engineer
engineer = NYISOFeatureEngineer()

# Engineer all features
featured_df = engineer.engineer_all_features(processed_df)

print(f"Featured records: {featured_df.count():,}")
print(f"Total features: {len(featured_df.columns)}")

In [ ]:
# List all feature columns
print("Feature columns:")
for col in sorted(featured_df.columns):
    print(f"  - {col}")

In [ ]:
# Check spike distribution
featured_df.groupBy("is_price_spike").count().show()

## 4. Model Training

In [ ]:
# Split data
train_df, test_df = featured_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training samples: {train_df.count():,}")
print(f"Test samples: {test_df.count():,}")

# Cache for faster training
train_df.cache()
test_df.cache()

In [ ]:
# Train Price Predictor
print("Training Price Predictor...")
price_model = PricePredictor()
price_model.train(train_df)
print("Done!")

In [ ]:
# Train Demand Forecaster
print("Training Demand Forecaster...")
demand_model = DemandForecaster()
demand_model.train(train_df)
print("Done!")

In [ ]:
# Train Spike Detector
print("Training Spike Detector...")
spike_model = SpikeDetector()
spike_model.train(train_df)
print("Done!")

## 5. Model Evaluation

In [ ]:
# Initialize evaluator
evaluator = ModelEvaluator()

# Evaluate Price Predictor
print("=" * 50)
print("Price Predictor Evaluation")
print("=" * 50)
price_result = evaluator.evaluate_model(price_model, test_df)
evaluator.print_summary(price_result)

In [ ]:
# Evaluate Demand Forecaster
print("=" * 50)
print("Demand Forecaster Evaluation")
print("=" * 50)
demand_result = evaluator.evaluate_model(demand_model, test_df)
evaluator.print_summary(demand_result)

In [ ]:
# Evaluate Spike Detector
print("=" * 50)
print("Spike Detector Evaluation")
print("=" * 50)
spike_result = evaluator.evaluate_model(spike_model, test_df)
evaluator.print_summary(spike_result)

In [ ]:
# Evaluate by hour
price_predictions = price_model.predict(test_df)
hourly_metrics = evaluator.evaluate_by_hour(
    price_model, price_predictions, "LBMP_avg", "predicted_price"
)

print("\nPrice Prediction MAPE by Hour:")
for hour, metrics in sorted(hourly_metrics.items()):
    print(f"  Hour {hour:02d}: {metrics['mape']:.2f}%")

In [ ]:
# Feature importance
importance = price_model.get_feature_importance()
if importance:
    print("\nTop 10 Features for Price Prediction:")
    sorted_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:10]
    for feat, imp in sorted_features:
        print(f"  {feat}: {imp:.4f}")

## 6. Save Models & Results

In [ ]:
# Save models
price_model.save()
demand_model.save()
spike_model.save()

# Save evaluation results
evaluator.save_results()

print("\nAll models and results saved!")

In [ ]:
# Cleanup
train_df.unpersist()
test_df.unpersist()

print("Done!")